# Transformer Decoder-Only para tutor de Teoria Musical


In [70]:
# Instale apenas se necessário.
# %pip install torch tokenizers tqdm numpy

import math
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tqdm.auto import tqdm

## 1. Configuração

In [71]:
@dataclass
class Config:
    tokenizer_path: str = "tokenizer/tokenizer.json"
    data_dir: str = "data"
    checkpoint_dir: str = "checkpoints"

    train_file: str = "teoria_musical_treino_100M.txt"
    val_file: str = "teoria_musical_validacao_5M.txt"
    test_file: str = "teoria_musical_teste_5M.txt"

    seed: int = 42

    block_size: int = 256
    batch_size: int = 16
    num_workers: int = 0

    max_steps: int = 5000
    eval_interval: int = 100
    eval_batches: int = 50
    log_interval: int = 10

    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 384
    dropout: float = 0.10

    learning_rate: float = 3e-4
    min_learning_rate: float = 3e-5
    warmup_steps: int = 200
    weight_decay: float = 0.10
    grad_clip: float = 1.0

    use_amp: bool = True


cfg = Config()

assert cfg.n_embd % cfg.n_head == 0
assert cfg.block_size >= 8
assert cfg.max_steps > 0

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
amp_enabled = cfg.use_amp and device == "cuda"

print("Device:", device)
print("AMP:", amp_enabled)

if device == "cuda":
    torch.set_float32_matmul_precision("high")

Device: cuda
AMP: True


## 2. Carregar o tokenizer

In [72]:
tokenizer_path = Path(cfg.tokenizer_path)

if not tokenizer_path.exists():
    raise FileNotFoundError(
        f"Tokenizer não encontrado em: {tokenizer_path.resolve()}\n"
        "Execute primeiro o notebook de treinamento do tokenizer."
    )

tokenizer = Tokenizer.from_file(str(tokenizer_path))

SPECIAL_TOKENS = [
    "<pad>", "<bos>", "<eos>", "<unk>",
    "<registro>", "</registro>",
    "<conceito>", "</conceito>",
    "<conteudo>", "</conteudo>",
    "<pergunta>", "</pergunta>",
    "<resposta>", "</resposta>",
    "<exercicio>", "</exercicio>",
    "<analise>", "</analise>",
]

missing_tokens = [token for token in SPECIAL_TOKENS if tokenizer.token_to_id(token) is None]

if missing_tokens:
    raise ValueError(
        "O tokenizer não contém todos os tokens especiais exigidos pelo corpus:\n"
        + "\n".join(f"- {token}" for token in missing_tokens)
        + "\n\nTreine novamente o tokenizer incluindo essas tags como special tokens."
    )

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
UNK_ID = tokenizer.token_to_id("<unk>")

vocab_size = tokenizer.get_vocab_size()

print("Vocabulário:", vocab_size)
print("PAD/BOS/EOS/UNK:", PAD_ID, BOS_ID, EOS_ID, UNK_ID)

Vocabulário: 1730
PAD/BOS/EOS/UNK: 0 1 2 3


## 3. Carregar treino, validação e teste sem reparticionamento

In [73]:
data_dir = Path(cfg.data_dir)

paths = {
    "train": data_dir / cfg.train_file,
    "val": data_dir / cfg.val_file,
    "test": data_dir / cfg.test_file,
}

for split_name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Arquivo de {split_name} não encontrado: {path.resolve()}")

texts = {
    split_name: path.read_text(encoding="utf-8").strip()
    for split_name, path in paths.items()
}

for split_name, text in texts.items():
    if not text:
        raise ValueError(f"O arquivo de {split_name} está vazio.")

    words = len(re.findall(r"\S+", text))
    print(f"{split_name:>5}: {paths[split_name].name} | {words:,} palavras | {len(text):,} caracteres")

train: teoria_musical_treino_100M.txt | 100,000,000 palavras | 715,553,533 caracteres
  val: teoria_musical_validacao_5M.txt | 5,000,000 palavras | 35,696,160 caracteres
 test: teoria_musical_teste_5M.txt | 5,000,000 palavras | 35,197,121 caracteres


## 4. Extrair registros completos

Cada unidade de treinamento corresponde, de preferência, a um bloco `<registro>...</registro>` inteiro. O texto antes do primeiro registro (cabeçalho) vira uma amostra à parte.

In [74]:
REGISTER_PATTERN = re.compile(r"<registro\b[^>]*>.*?</registro>", flags=re.IGNORECASE | re.DOTALL)


def extract_samples(text: str) -> list[dict]:
    samples = []
    matches = list(REGISTER_PATTERN.finditer(text))

    if not matches:
        raise ValueError(
            "Nenhum bloco <registro>...</registro> foi encontrado. "
            "Confirme se os arquivos correspondem ao corpus estruturado."
        )

    prefix = text[:matches[0].start()].strip()
    if prefix:
        samples.append({"type": "cabecalho", "text": prefix})

    for match in matches:
        record = match.group(0).strip()

        if "<pergunta>" in record and "<resposta>" in record:
            sample_type = "pergunta_resposta"
        elif "<exercicio>" in record:
            sample_type = "exercicio"
        elif "<analise>" in record:
            sample_type = "analise"
        elif "<conceito>" in record and "<conteudo>" in record:
            sample_type = "conceito_conteudo"
        else:
            sample_type = "outro"

        samples.append({"type": sample_type, "text": record})

    return samples


samples = {split_name: extract_samples(text) for split_name, text in texts.items()}

for split_name, split_samples in samples.items():
    print(f"\n{split_name.upper()}: {len(split_samples):,} registros")
    print(Counter(item["type"] for item in split_samples))


TRAIN: 1,237,873 registros
Counter({'pergunta_resposta': 1237871, 'cabecalho': 1, 'conceito_conteudo': 1})

VAL: 61,894 registros
Counter({'pergunta_resposta': 61892, 'cabecalho': 1, 'conceito_conteudo': 1})

TEST: 61,514 registros
Counter({'pergunta_resposta': 61512, 'cabecalho': 1, 'conceito_conteudo': 1})


## 5. Verificar vazamento literal entre os conjuntos

In [75]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


normalized_sets = {
    split_name: {normalize_text(item["text"]) for item in split_samples}
    for split_name, split_samples in samples.items()
}

overlaps = {
    "train_val": len(normalized_sets["train"] & normalized_sets["val"]),
    "train_test": len(normalized_sets["train"] & normalized_sets["test"]),
    "val_test": len(normalized_sets["val"] & normalized_sets["test"]),
}

print(overlaps)

if any(overlaps.values()):
    raise ValueError("Foram encontrados registros literalmente repetidos entre os conjuntos.")

{'train_val': 0, 'train_test': 0, 'val_test': 0}


## 6. Tokenização e dataset causal

In [76]:
def tokenize_text(text: str) -> list[int]:
    return [BOS_ID] + tokenizer.encode(text).ids + [EOS_ID]


class StructuredCausalDataset(Dataset):
    def __init__(self, samples: list[dict], block_size: int, training: bool):
        self.block_size = block_size
        self.training = training
        self.examples = []

        for sample in samples:
            ids = tokenize_text(sample["text"])

            if len(ids) <= block_size + 1:
                self.examples.append({"ids": ids, "type": sample["type"]})
                continue

            stride = max(1, block_size // 2)
            for start in range(0, len(ids) - 1, stride):
                window = ids[start:start + block_size + 1]
                if len(window) < 2:
                    continue

                self.examples.append({"ids": window, "type": sample["type"]})

                if start + block_size + 1 >= len(ids):
                    break

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        ids = example["ids"]

        if self.training and len(ids) > self.block_size + 1:
            max_start = len(ids) - self.block_size - 1
            start = random.randint(0, max_start)
            ids = ids[start:start + self.block_size + 1]

        return torch.tensor(ids, dtype=torch.long)


def causal_collate_fn(batch):
    max_len = min(max(len(ids) for ids in batch), cfg.block_size + 1)
    padded = torch.full((len(batch), max_len), fill_value=PAD_ID, dtype=torch.long)

    for i, ids in enumerate(batch):
        ids = ids[:max_len]
        padded[i, :len(ids)] = ids

    x = padded[:, :-1]
    y = padded[:, 1:]
    return x, y


train_ds = StructuredCausalDataset(samples["train"], cfg.block_size, training=True)
val_ds = StructuredCausalDataset(samples["val"], cfg.block_size, training=False)
test_ds = StructuredCausalDataset(samples["test"], cfg.block_size, training=False)

generator = torch.Generator()
generator.manual_seed(cfg.seed)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=causal_collate_fn,
    num_workers=cfg.num_workers,
    pin_memory=(device == "cuda"),
    generator=generator,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=causal_collate_fn,
    num_workers=cfg.num_workers,
    pin_memory=(device == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=causal_collate_fn,
    num_workers=cfg.num_workers,
    pin_memory=(device == "cuda"),
)

print("Exemplos de treino:", len(train_ds))
print("Exemplos de validação:", len(val_ds))
print("Exemplos de teste:", len(test_ds))
print("Batches de treino:", len(train_loader))

Exemplos de treino: 1237876
Exemplos de validação: 61897
Exemplos de teste: 61517
Batches de treino: 77367


## 7. Modelo decoder-only com RMSNorm, SwiGLU, atenção causal e KV cache

In [77]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout = cfg.dropout

        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, kv_cache: Optional[dict] = None, use_cache: bool = False):
        batch_size, seq_len, emb_dim = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        past_len = 0

        if use_cache and kv_cache is not None:
            if "k" in kv_cache and "v" in kv_cache:
                past_len = kv_cache["k"].size(2)
                k = torch.cat([kv_cache["k"], k], dim=2)
                v = torch.cat([kv_cache["v"], v], dim=2)

            kv_cache["k"] = k
            kv_cache["v"] = v

        if not use_cache:
            attn_mask = None
            is_causal = True
        elif past_len == 0 and seq_len > 1:
            # primeira passagem, com o prompt inteiro
            attn_mask = None
            is_causal = True
        else:
            # geração incremental (seq_len == 1 na prática): a posição nova
            # pode olhar pra tudo que já foi cacheado
            attn_mask = None
            is_causal = False

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, emb_dim)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        hidden_dim = int(8 * cfg.n_embd / 3)
        hidden_dim = 64 * math.ceil(hidden_dim / 64)

        self.gate_proj = nn.Linear(cfg.n_embd, hidden_dim, bias=False)
        self.up_proj = nn.Linear(cfg.n_embd, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, cfg.n_embd, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.norm1 = RMSNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, kv_cache: Optional[dict] = None, use_cache: bool = False):
        x = x + self.attn(self.norm1(x), kv_cache=kv_cache, use_cache=use_cache)
        x = x + self.mlp(self.norm2(x))
        return x

In [78]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg: Config, vocab_size: int):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = vocab_size

        self.token_emb = nn.Embedding(vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.dropout = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layer)])

        self.norm_f = RMSNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, kv_caches: Optional[list] = None, use_cache: bool = False):
        batch_size, seq_len = idx.shape
        past_len = 0

        if use_cache and kv_caches:
            first_cache = kv_caches[0]
            if first_cache and "k" in first_cache:
                past_len = first_cache["k"].size(2)

        if past_len + seq_len > self.cfg.block_size:
            raise ValueError(f"Contexto total {past_len + seq_len} excede block_size={self.cfg.block_size}.")

        positions = torch.arange(past_len, past_len + seq_len, device=idx.device)

        x = self.token_emb(idx)
        x = x + self.pos_emb(positions)[None, :, :]
        x = self.dropout(x)

        if kv_caches is None:
            kv_caches = [None] * len(self.blocks)

        for block, cache in zip(self.blocks, kv_caches):
            x = block(x, kv_cache=cache, use_cache=use_cache)

        x = self.norm_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=PAD_ID,
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        prompt_ids: list[int],
        max_new_tokens: int = 160,
        temperature: float = 0.7,
        top_k: Optional[int] = 50,
        top_p: Optional[float] = 0.95,
    ):
        self.eval()

        max_prompt_len = max(1, self.cfg.block_size - max_new_tokens)
        prompt_ids = prompt_ids[-max_prompt_len:]

        idx = torch.tensor([prompt_ids], dtype=torch.long, device=next(self.parameters()).device)

        kv_caches = [dict() for _ in self.blocks]
        generated = list(prompt_ids)

        logits, _ = self(idx, kv_caches=kv_caches, use_cache=True)
        next_logits = logits[:, -1, :]

        for _ in range(max_new_tokens):
            logits_step = next_logits / max(temperature, 1e-6)

            if top_k is not None:
                k = min(top_k, logits_step.size(-1))
                threshold = torch.topk(logits_step, k=k).values[:, -1, None]
                logits_step = torch.where(
                    logits_step < threshold,
                    torch.full_like(logits_step, float("-inf")),
                    logits_step,
                )

            if top_p is not None and 0 < top_p < 1:
                sorted_logits, sorted_indices = torch.sort(logits_step, descending=True)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                remove_mask = cumulative_probs > top_p
                remove_mask[:, 1:] = remove_mask[:, :-1].clone()
                remove_mask[:, 0] = False

                sorted_logits = sorted_logits.masked_fill(remove_mask, float("-inf"))
                logits_step = torch.full_like(logits_step, float("-inf")).scatter(1, sorted_indices, sorted_logits)

            probs = F.softmax(logits_step, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

            token_id = int(next_id.item())
            generated.append(token_id)

            if token_id == EOS_ID:
                break
            if len(generated) >= self.cfg.block_size:
                break

            logits, _ = self(next_id, kv_caches=kv_caches, use_cache=True)
            next_logits = logits[:, -1, :]

        return generated

## 8. Inicializar modelo, otimizador, scheduler e AMP

In [79]:
model = DecoderOnlyTransformer(cfg, vocab_size).to(device)

n_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Parâmetros: {n_params / 1e6:.2f} milhões")

# AdamW aplica weight decay a cada optimizer.step(). Matrizes recebem
# regularização; bias e normas ficam de fora (senão penaliza escala/deslocamento).
decay_params = []
no_decay_params = []

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if parameter.ndim >= 2:
        decay_params.append(parameter)
    else:
        no_decay_params.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=cfg.learning_rate,
    betas=(0.9, 0.95),
)

print("Weight decay ativo durante todo o treino:", cfg.weight_decay)
print("Parâmetros regularizados:", sum(p.numel() for p in decay_params))
print("Parâmetros sem regularização:", sum(p.numel() for p in no_decay_params))


def get_learning_rate(step: int) -> float:
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / max(1, cfg.warmup_steps)

    progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return cfg.min_learning_rate + cosine * (cfg.learning_rate - cfg.min_learning_rate)


scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

Parâmetros: 11.38 milhões
Weight decay ativo durante todo o treino: 0.1
Parâmetros regularizados: 11379456
Parâmetros sem regularização: 4992


## 9. Avaliação

No treino, só treino e validação são avaliados a cada `eval_interval`. O teste fica reservado pra depois, quando o melhor checkpoint já estiver carregado.

In [80]:
@torch.no_grad()
def evaluate_loader(loader, max_batches: Optional[int] = None):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch_index, (x, y) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device, enabled=amp_enabled):
            _, loss = model(x, y)

        valid_tokens = (y != PAD_ID).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    model.train()

    if total_tokens == 0:
        return float("nan")

    return total_loss / total_tokens


@torch.no_grad()
def estimate_train_val():
    return {
        "train": evaluate_loader(train_loader, cfg.eval_batches),
        "val": evaluate_loader(val_loader, cfg.eval_batches),
    }


initial_losses = estimate_train_val()

print("Loss inicial:", initial_losses)
print("Perplexidade inicial de validação:", math.exp(min(initial_losses["val"], 20)))

Loss inicial: {'train': 7.529336179309379, 'val': 7.517205331430339}
Perplexidade inicial de validação: 1839.4195363007746


## 10. Treinamento e seleção do melhor checkpoint

In [81]:
checkpoint_dir = Path(cfg.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

last_checkpoint_path = checkpoint_dir / "decoder_only_musica_last.pt"
best_checkpoint_path = checkpoint_dir / "decoder_only_musica_best.pt"

model.train()

step = 0
best_val_loss = float("inf")
history = []

train_iterator = iter(train_loader)
pbar = tqdm(total=cfg.max_steps, desc="Treinando")

while step < cfg.max_steps:
    try:
        x, y = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        x, y = next(train_iterator)

    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

    learning_rate = get_learning_rate(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = learning_rate

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(device_type=device, enabled=amp_enabled):
        _, loss = model(x, y)

    scaler.scale(loss).backward()

    if cfg.grad_clip is not None:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

    scaler.step(optimizer)
    scaler.update()

    step += 1
    pbar.update(1)

    if step % cfg.log_interval == 0:
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{learning_rate:.2e}")

    should_evaluate = step == 1 or step % cfg.eval_interval == 0 or step == cfg.max_steps

    if should_evaluate:
        losses = estimate_train_val()

        record = {
            "step": step,
            "train_loss": losses["train"],
            "val_loss": losses["val"],
            "val_perplexity": math.exp(min(losses["val"], 20)),
            "learning_rate": learning_rate,
        }
        history.append(record)

        print(
            f"\nstep {step}: train={losses['train']:.4f} | val={losses['val']:.4f} | "
            f"val ppl={record['val_perplexity']:.2f} | lr={learning_rate:.2e}"
        )

        checkpoint = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "config": asdict(cfg),
            "vocab_size": vocab_size,
            "step": step,
            "losses": losses,
            "history": history,
        }

        torch.save(checkpoint, last_checkpoint_path)

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save(checkpoint, best_checkpoint_path)
            print(f"Novo melhor checkpoint salvo. val loss={best_val_loss:.4f}")
        else:
            print(f"A validação não melhorou, mas o treinamento continua até max_steps={cfg.max_steps}.")

pbar.close()
print("Treinamento encerrado no step:", step)

Treinando:   0%|          | 0/5000 [00:00<?, ?it/s]


step 1: train=7.5187 | val=7.5054 | val ppl=1817.82 | lr=1.50e-06
Novo melhor checkpoint salvo. val loss=7.5054

step 100: train=2.6923 | val=2.6750 | val ppl=14.51 | lr=1.50e-04
Novo melhor checkpoint salvo. val loss=2.6750

step 200: train=0.6195 | val=0.6634 | val ppl=1.94 | lr=3.00e-04
Novo melhor checkpoint salvo. val loss=0.6634

step 300: train=0.3056 | val=0.3686 | val ppl=1.45 | lr=3.00e-04
Novo melhor checkpoint salvo. val loss=0.3686

step 400: train=0.2435 | val=0.3311 | val ppl=1.39 | lr=2.99e-04
Novo melhor checkpoint salvo. val loss=0.3311

step 500: train=0.2169 | val=0.3080 | val ppl=1.36 | lr=2.97e-04
Novo melhor checkpoint salvo. val loss=0.3080

step 600: train=0.2084 | val=0.3116 | val ppl=1.37 | lr=2.95e-04
A validação não melhorou, mas o treinamento continuará até max_steps=5000.

step 700: train=0.1973 | val=0.2921 | val ppl=1.34 | lr=2.93e-04
Novo melhor checkpoint salvo. val loss=0.2921

step 800: train=0.1904 | val=0.3203 | val ppl=1.38 | lr=2.90e-04
A valid

## 11. Carregar o melhor checkpoint e avaliar o teste uma única vez

In [82]:
if not best_checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint não encontrado: {best_checkpoint_path.resolve()}")

checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Melhor checkpoint carregado no step:", checkpoint["step"])

final_train_loss = evaluate_loader(train_loader, max_batches=None)
final_val_loss = evaluate_loader(val_loader, max_batches=None)
final_test_loss = evaluate_loader(test_loader, max_batches=None)

print(f"Train loss: {final_train_loss:.4f}")
print(f"Val loss:   {final_val_loss:.4f}")
print(f"Test loss:  {final_test_loss:.4f}")
print("Test perplexity:", math.exp(min(final_test_loss, 20)))

Melhor checkpoint carregado no step: 4200
Train loss: 0.1291
Val loss:   0.2140
Test loss:  1.5379
Test perplexity: 4.654736562323127


## 12. Inferência estruturada

In [83]:
def encode_prompt(text: str) -> list[int]:
    return [BOS_ID] + tokenizer.encode(text).ids


def decode_output(ids: list[int]) -> str:
    filtered = [token_id for token_id in ids if token_id not in {PAD_ID, BOS_ID, EOS_ID}]
    return tokenizer.decode(filtered, skip_special_tokens=False)


def gerar_resposta(pergunta: str, max_new_tokens: int = 180, temperature: float = 0.4, top_k: int = 50, top_p: float = 0.95):
    prompt = f"<registro>\n<pergunta>{pergunta}</pergunta>\n<resposta>"

    generated_ids = model.generate(
        prompt_ids=encode_prompt(prompt),
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )

    return decode_output(generated_ids)


print(gerar_resposta("O que é andamento musical?"))

<registro>
<pergunta>O que é andamento musical?</pergunta>
<resposta>" tipo="de-37494669registro4476341687registro id="de00061248 perfeita79ponciona="de?</pergunta>
<resposta>A análise começa pela identificação da função musical. Andamento é a mesma altura. Para reconhecer o recurso, marque o pulso, identifique a subdivisão e observe a posição dos ataques e acentos. O exemplo deve confirmar a regra sem introduzir notas estranhas. Na variação 15, preserve a mesma regra ao aplicar o conceito em outro exemplo.</resposta>
</registro>


## 13. Avaliação qualitativa com prompts fixos

In [84]:
@torch.no_grad()
def avaliar_prompts(perguntas: list[str], max_new_tokens: int = 180, temperature: float = 0.35, top_k: int = 50, top_p: float = 0.95):
    model.eval()

    for index, pergunta in enumerate(perguntas, start=1):
        saida = gerar_resposta(
            pergunta=pergunta,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        print("=" * 80)
        print(f"AVALIAÇÃO {index}")
        print("=" * 80)
        print(saida)
        print()


perguntas_avaliacao = [
    "O que é campo harmônico?",
    "Monte o campo harmônico maior de Dó.",
    "Monte o campo harmônico maior de Fá.",
    "Por que em Fá maior usamos Si bemol e não Lá sustenido?",
    "Monte uma progressão ii-V-I em Ré maior.",
    "O que são dominantes secundárias?",
    "Qual é a diferença entre tonicização e modulação?",
    "Como funciona uma cadência autêntica perfeita?",
]

avaliar_prompts(perguntas_avaliacao)

AVALIAÇÃO 1
<registro>
<pergunta>O que é campo harmônico?</pergunta>
<resposta>" tipon819262645registroregistro id="de86, escuta6 explicação de006 central é o conceito deve ser entendido pela definição e pela aplicação. A tríade aumentadainâmica é o construídos sobre os graus de uma escala. Sua identificação depende do comportamento no contexto musical, e não apenas do nome. Uma resposta adequada deve apresentar a regra, um exemplo e a relação com conceitos próximos. A resposta está completa quando regra e exemplo permanecem coerentes. Na variação 49, preserve a mesma regra ao aplicar o conceito em outro exemplo.</resposta>
</registro>

AVALIAÇÃO 2
<registro>
<pergunta>Monte o campo harmônico maior de Dó.</pergunta>
<resposta>A exp0025441561761236054168491264447847618487="ri6290062604470736634072806300061661370000007570095700006490915929064604290082900092500072900827007460092900""""""""""""""""""""""""""""""""""""""",,,,,,,,,,,,,,,,,,,,,,,,,,,

AVALIAÇÃO 3
<registro>
<pergunta>Monte o 